In [0]:
%sql
select * from workspace.bronze.finnhub_company_news;

In [0]:
%sql
-- Silver: parse raw news JSON, derive published_at, and drop duplicate articles.

-- Step 1: Define the table schema (with comments) - no AS SELECT allowed here alongside an explicit column list.
CREATE OR REPLACE TABLE silver_finnhub_company_news
(
  symbol STRING COMMENT "Stock ticker symbol the news relates to.",
  news_id BIGINT COMMENT "Unique identifier for the news article.",
  headline STRING COMMENT "Headline of the news article.",
  source STRING COMMENT "Publisher or news source.",
  published_at TIMESTAMP COMMENT "Timestamp when the article was published.",
  category STRING COMMENT "Category assigned to the article.",
  summary STRING COMMENT "Short summary of the article.",
  url STRING COMMENT "Link to the full article.",
  ingested_at STRING COMMENT "Timestamp when the record was ingested into bronze."
)
COMMENT "Cleaned company news data for the finnhub medallion pipeline.";

-- Step 2: Populate it. INSERT OVERWRITE fully refreshes the table each run,
-- matching the "recompute everything" behavior your original CREATE OR REPLACE was going for.
INSERT OVERWRITE TABLE silver_finnhub_company_news
SELECT
  symbol,
  news_item.id AS news_id,
  news_item.headline AS headline,
  news_item.source AS source,
  CAST(from_unixtime(news_item.datetime) AS TIMESTAMP) AS published_at,
  news_item.category AS category,
  news_item.summary AS summary,
  news_item.url AS url,
  ingested_at
FROM (
  SELECT
    symbol,
    ingested_at,
    explode(
      from_json(
        raw_payload,
        "array<struct<category:string,datetime:bigint,headline:string,id:bigint,image:string,related:string,source:string,summary:string,url:string>>"
      )
    ) AS news_item
  FROM workspace.bronze.finnhub_company_news
  WHERE status_code = 200
)
QUALIFY ROW_NUMBER() OVER (PARTITION BY symbol, news_item.id ORDER BY ingested_at DESC) = 1;

In [0]:
%sql
select * from silver_finnhub_company_news;